# calibrate.ipynb — Heston DDN 模型校准

本 Notebook 负责：
1. 加载已训练的 `HestonDDN` 权重
2. 构建单参数期权链（模拟市场数据）
3. 运行 Multi-start Adam 校准
4. 对比校准值 vs Ground Truth
5. 用校准后参数重新定价验证

In [1]:
## Section 8: 导入依赖与项目路径配置
import sys
import os
from pathlib import Path

PROJECT_ROOT = Path.cwd()   # 运行时当前目录 = Heston_Project/
sys.path.insert(0, str(PROJECT_ROOT))

from modules.model       import HestonDDN
from modules.calibration import HestonCalibrator
from modules.pricing     import calculate_heston_price

import numpy as np
import pandas as pd
import torch

# ---- 路径常量 ----
MODEL_PATH = PROJECT_ROOT / "models" / "heston_ddn_weights.pth"

# 兼容：若本项目 models/ 下无权重，则尝试上级目录
FALLBACK_MODEL = Path("/Users/liaojiansong/calibration/heston_ddn_weights.pth")

print(f"PROJECT_ROOT : {PROJECT_ROOT}")
print(f"MODEL_PATH   : {MODEL_PATH}")

PROJECT_ROOT : /Users/liaojiansong/calibration/Zhang_realize
MODEL_PATH   : /Users/liaojiansong/calibration/Zhang_realize/models/heston_ddn_weights.pth


In [2]:
## Section 9: 加载已训练模型权重

device = torch.device(
    "mps"  if torch.backends.mps.is_available() else
    "cuda" if torch.cuda.is_available() else "cpu"
)
print(f"使用设备: {device}")

# 确定权重文件路径
if MODEL_PATH.exists():
    ckpt_path = MODEL_PATH
elif FALLBACK_MODEL.exists():
    ckpt_path = FALLBACK_MODEL
    print(f"⚠️  使用备用权重: {ckpt_path}")
else:
    raise FileNotFoundError(
        f"未找到模型权重。请先运行 01_train.ipynb，或将权重放置于: {MODEL_PATH}"
    )

# 加载
ckpt  = torch.load(ckpt_path, map_location=device)
model = HestonDDN(
    input_dim   = ckpt.get('input_dim',  9),
    heston_param_dim = ckpt.get('heston_dim', 5),
).to(device)
model.load_state_dict(ckpt['model_state_dict'])
model.is_fitted = ckpt.get('is_fitted', True)
model.eval()

print(f"✅ 权重已加载: {ckpt_path}")
print(f"   is_fitted = {model.is_fitted}")
print(f"   y_min = {model.y_min.item():.6f}")
print(f"   y_max = {model.y_max.item():.6f}")
print(f"   X_min = {model.X_min.cpu().numpy()}")
print(f"   X_max = {model.X_max.cpu().numpy()}")

使用设备: mps
✅ 权重已加载: /Users/liaojiansong/calibration/Zhang_realize/models/heston_ddn_weights.pth
   is_fitted = True
   y_min = 0.000000
   y_max = 0.556161
   X_min = [ 1.0011792e-02  1.0004548e-02  1.0000127e-01 -8.9999771e-01
  1.0001554e-02  1.0003702e-03  5.0008450e-02  1.0028770e+01
 -4.9999633e-01]
   X_max = [ 4.9999971e+00  9.9999273e-01  9.9998772e-01 -5.0001502e-02
  9.9998927e-01  9.9999733e-02  9.9999821e-01  5.9999736e+03
  4.9999893e-01]


## 市场数据模拟校准

In [3]:
## Section 10: 构建单参数期权链（市场数据模拟）

# ---- Ground Truth Heston 参数 ----
true_kappa  = 2.0
true_lambda = 0.1
true_sigma  = 0.4
true_rho    = -0.6
true_v0     = 0.05
true_r      = 0.05
true_S0     = 1000.0

true_heston = np.array([true_kappa, true_lambda, true_sigma, true_rho, true_v0])

print("Ground Truth Heston 参数:")
for n, v in zip(['kappa','lambda','sigma','rho','v0'], true_heston):
    print(f"  {n:>8s} = {v:.4f}")
print(f"  {'r':>8s} = {true_r}")
print(f"  {'S0':>8s} = {true_S0}")
print()

# ---- 构建期权链网格 ----
taus      = [0.3, 0.4, 0.5, 0.6, 0.7]
moneyness = np.linspace(0.6, 1.5, 20)

rows = []
for tau in taus:
    for m in moneyness:
        K    = true_S0 * m
        Pmkt = calculate_heston_price(
            true_kappa, true_lambda, true_sigma, true_rho, true_v0,
            true_r, tau, true_S0, K
        )
        if not np.isnan(Pmkt) and Pmkt > 0.01:
            rows.append({'r': true_r, 'tau': tau, 'S0': true_S0, 'K': K, 'P_mkt': Pmkt})

sample_df = pd.DataFrame(rows)
print(f"✅ 有效期权数: {len(sample_df)}")
print(sample_df[['tau','K','P_mkt']].head(10).to_string(index=False))

Ground Truth Heston 参数:
     kappa = 2.0000
    lambda = 0.1000
     sigma = 0.4000
       rho = -0.6000
        v0 = 0.0500
         r = 0.05
        S0 = 1000.0

✅ 有效期权数: 99
 tau           K      P_mkt
 0.3  600.000000 408.976956
 0.3  647.368421 362.472633
 0.3  694.736842 316.206185
 0.3  742.105263 270.435318
 0.3  789.473684 225.604390
 0.3  836.842105 182.404284
 0.3  884.210526 141.798308
 0.3  931.578947 104.973464
 0.3  978.947368  73.180175
 0.3 1026.315789  47.454416


In [14]:
## Section 11: Multi-start Adam 校准执行
import time
market_data = {
    'r':     sample_df['r'].values.astype(np.float32),
    'tau':   sample_df['tau'].values.astype(np.float32),
    'S0':    sample_df['S0'].values.astype(np.float32),
    'K':     sample_df['K'].values.astype(np.float32),
    'P_mkt': sample_df['P_mkt'].values.astype(np.float32),
}

calibrator = HestonCalibrator(model, device, seed=0)

t0 = time.time()
best_theta, best_loss, all_results = calibrator.calibrate(
    market_data,
    n_starts  = 10,
    lr        = 5e-3,
    max_steps = 500,
    patience  = 40,
    verbose   = True,
)
print(f"校准完成! 最佳参数: {best_theta}, 最佳损失: {best_loss:.6f}, 耗时: {time.time() - t0:.2f} 秒")

initial guess: [[ 3.18843882  0.27708885  0.13687617 -0.88595151  0.81513754]
 [ 4.56465033  0.61056942  0.7565469  -0.43791876  0.9357217 ]
 [ 4.08110924  0.01271112  0.87166385 -0.87145226  0.73235889]
 [ 0.88652155  0.86454713  0.5873151  -0.64524489  0.42846035]
 [ 0.15131516  0.13304044  0.70356197 -0.34988892  0.61923126]
 [ 1.924551    0.99723784  0.9827518  -0.31728931  0.65395468]
 [ 3.44534919  0.39503221  0.22158685 -0.28673491  0.53010078]
 [ 1.55810696  0.49097701  0.90053905 -0.10606301  0.36421724]
 [ 2.86193386  0.3286507   0.63487003 -0.61277546  0.39770281]
 [ 4.45246902  0.23488602  0.66086843 -0.82858696  0.83431771]]

  市场期权数   : 99
  初始点数     : 10
  Adam lr      : 0.005，max_steps=500，patience=40
  price_norm   : [0.0000, 0.4226]

  [start  1/10] step= 100 | loss(norm)=1.0350e-03 | lr=5.00e-03
  [start  1/10] step= 100 | loss(norm)=1.0350e-03 | lr=5.00e-03
  [start  1/10] step= 200 | loss(norm)=7.6630e-05 | lr=5.00e-03
  [start  1/10] step= 200 | loss(norm)=7.6630e

In [15]:
## Section 12: 校准结果 vs Ground Truth 对比

param_names = ['kappa', 'lambda', 'sigma', 'rho', 'v0']
THRESHOLD   = 10.0  # 相对误差超过此阈值时高亮

print("=" * 62)
print("校准结果 vs Ground Truth")
print("=" * 62)
print(f"  {'参数':>8s} | {'真实值':>10s} | {'校准值':>10s} | {'绝对误差':>10s} | {'相对误差':>8s}")
print("  " + "-" * 58)

for n, tv, cv in zip(param_names, true_heston, best_theta):
    ae   = abs(cv - tv)
    re   = ae / (abs(tv) + 1e-8) * 100
    flag = "⚠️ " if re > THRESHOLD else "   "
    print(f"  {flag}{n:>6s} | {tv:10.6f} | {cv:10.6f} | {ae:10.6f} | {re:7.2f}%")

print()
print(f"  最终 normalized MSE: {best_loss:.6e}")

校准结果 vs Ground Truth
        参数 |        真实值 |        校准值 |       绝对误差 |     相对误差
  ----------------------------------------------------------
  ⚠️  kappa |   2.000000 |   3.588379 |   1.588379 |   79.42%
  ⚠️ lambda |   0.100000 |   0.085829 |   0.014171 |   14.17%
  ⚠️  sigma |   0.400000 |   0.713949 |   0.313949 |   78.49%
        rho |  -0.600000 |  -0.543860 |   0.056140 |    9.36%
  ⚠️     v0 |   0.050000 |   0.040037 |   0.009963 |   19.93%

  最终 normalized MSE: 4.044668e-06


In [16]:
## Section 13: 用校准后参数重新定价验证

M = len(sample_df)

model.eval()
with torch.no_grad():
    cal_theta    = torch.tensor(best_theta, dtype=torch.float32).to(device)
    cal_expanded = cal_theta.unsqueeze(0).expand(M, -1)          # (M, 5)

    S0_t   = torch.tensor(sample_df['S0'].values,  dtype=torch.float32).to(device)
    r_t    = torch.tensor(sample_df['r'].values,   dtype=torch.float32).to(device)
    tau_t  = torch.tensor(sample_df['tau'].values, dtype=torch.float32).to(device)
    lks_t  = torch.tensor(
        np.log(sample_df['K'].values / sample_df['S0'].values),
        dtype=torch.float32
    ).to(device)

    mkt_feat = torch.stack([r_t, tau_t, S0_t, lks_t], dim=1)     # (M, 4)
    X_cal    = torch.cat([cal_expanded, mkt_feat], dim=1)          # (M, 9)

    P_cal_norm = model(X_cal)                                      # price_norm
    P_cal_abs  = (P_cal_norm * S0_t.view(-1, 1)).cpu().numpy()     # 还原绝对价格

P_mkt_arr = sample_df['P_mkt'].values
rel_errs  = np.abs(P_cal_abs.flatten() - P_mkt_arr) / (np.abs(P_mkt_arr) + 1e-8) * 100

# 前 10 条明细
print(f"  {'#':>4s} | {'市场价(绝对)':>13s} | {'校准价(绝对)':>13s} | {'偏差%':>7s}")
print("  " + "-"*46)
for i in range(M):
    flag = "✅" if rel_errs[i] < 5.0 else "❌"
    print(f"  {flag}{i+1:3d} | {P_mkt_arr[i]:13.4f} | {P_cal_abs[i,0]:13.4f} | {rel_errs[i]:6.2f}%")
mre = rel_errs.mean()
print()
print(f"  📊 全量 ({M} 条) 重定价统计:")
print(f"     平均相对误差 : {rel_errs.mean():.4f}%")
print(f"     最大相对误差 : {rel_errs.max():.4f}%")
print(f"     < 5% 占比    : {(rel_errs < 5.0).mean()*100:.1f}%")
print(f"MRE: {mre:.4f}%")
print("✅ 校准全流程完成！")

     # |       市场价(绝对) |       校准价(绝对) |     偏差%
  ----------------------------------------------
  ✅  1 |      408.9770 |      410.5502 |   0.38%
  ✅  2 |      362.4726 |      364.3813 |   0.53%
  ✅  3 |      316.2062 |      318.7061 |   0.79%
  ✅  4 |      270.4353 |      273.4744 |   1.12%
  ✅  5 |      225.6044 |      229.0067 |   1.51%
  ✅  6 |      182.4043 |      185.9485 |   1.94%
  ✅  7 |      141.7983 |      145.1437 |   2.36%
  ✅  8 |      104.9735 |      107.5431 |   2.45%
  ✅  9 |       73.1802 |       74.3792 |   1.64%
  ✅ 10 |       47.4544 |       47.3692 |   0.18%
  ✅ 11 |       28.2817 |       27.7678 |   1.82%
  ✅ 12 |       15.3477 |       15.2598 |   0.57%
  ❌ 13 |        7.5584 |        8.2196 |   8.75%
  ❌ 14 |        3.3944 |        4.6789 |  37.84%
  ❌ 15 |        1.4075 |        3.0832 | 119.05%
  ❌ 16 |        0.5480 |        2.4320 | 343.81%
  ❌ 17 |        0.2038 |        2.1864 | 972.85%
  ❌ 18 |        0.0735 |        2.1005 | 2758.38%
  ❌ 19 |        0.0

In [2]:

## Section 13b: Zhang 方法合成数据 —— IV 空间 MRE 计算（自包含）

import sys
import os
import time
import numpy as np
import pandas as pd
import torch
from pathlib import Path
from scipy.optimize import brentq
from scipy.stats import norm

# ── 路径 & 模型加载 ─────────────────────────────────────────────
PROJECT_ROOT = Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))
from modules.model       import HestonDDN
from modules.calibration import HestonCalibrator
from modules.pricing     import calculate_heston_price

MODEL_PATH = PROJECT_ROOT / "models" / "heston_ddn_weights.pth"
device = torch.device(
    "mps"  if torch.backends.mps.is_available() else
    "cuda" if torch.cuda.is_available() else "cpu"
)
ckpt  = torch.load(MODEL_PATH, map_location=device)
model_iv = HestonDDN(
    input_dim        = ckpt.get('input_dim', 9),
    heston_param_dim = ckpt.get('heston_dim', 5),
).to(device)
model_iv.load_state_dict(ckpt['model_state_dict'])
model_iv.is_fitted = ckpt.get('is_fitted', True)
model_iv.eval()
print(f"✅ 模型已加载 (device={device})")

# ── 合成数据参数（与 Section 10 保持一致）──────────────────────
true_kappa  = 2.0
true_lambda = 0.1
true_sigma  = 0.4
true_rho    = -0.6
true_v0     = 0.05
true_r      = 0.05
true_S0     = 1000.0
true_heston = np.array([true_kappa, true_lambda, true_sigma, true_rho, true_v0])

taus      = [0.3, 0.4, 0.5, 0.6, 0.7]
moneyness = np.linspace(0.6, 1.5, 20)
rows = []
for tau in taus:
    for m in moneyness:
        K    = true_S0 * m
        Pmkt = calculate_heston_price(
            true_kappa, true_lambda, true_sigma, true_rho, true_v0,
            true_r, tau, true_S0, K
        )
        if not np.isnan(Pmkt) and Pmkt > 0.01:
            rows.append({'r': true_r, 'tau': tau, 'S0': true_S0, 'K': K, 'P_mkt': Pmkt})
sample_df = pd.DataFrame(rows)
print(f"✅ 合成数据期权数: {len(sample_df)}")

# ── Multi-start Adam 校准 ───────────────────────────────────────
market_data = {
    'r':     sample_df['r'].values.astype(np.float32),
    'tau':   sample_df['tau'].values.astype(np.float32),
    'S0':    sample_df['S0'].values.astype(np.float32),
    'K':     sample_df['K'].values.astype(np.float32),
    'P_mkt': sample_df['P_mkt'].values.astype(np.float32),
}
calibrator = HestonCalibrator(model_iv, device, seed=0)
t0 = time.time()
best_theta_iv, best_loss_iv, _ = calibrator.calibrate(
    market_data, n_starts=10, lr=5e-3, max_steps=500, patience=40, verbose=False,
)
print(f"✅ 校准完成，耗时 {time.time()-t0:.2f}s，loss={best_loss_iv:.6e}")
print(f"   校准参数: {best_theta_iv}")

# ── 用校准参数重定价 ────────────────────────────────────────────
M = len(sample_df)
model_iv.eval()
with torch.no_grad():
    cal_theta    = torch.tensor(best_theta_iv, dtype=torch.float32).to(device)
    cal_expanded = cal_theta.unsqueeze(0).expand(M, -1)
    S0_t   = torch.tensor(sample_df['S0'].values,  dtype=torch.float32).to(device)
    r_t    = torch.tensor(sample_df['r'].values,   dtype=torch.float32).to(device)
    tau_t  = torch.tensor(sample_df['tau'].values, dtype=torch.float32).to(device)
    lks_t  = torch.tensor(np.log(sample_df['K'].values / sample_df['S0'].values), dtype=torch.float32).to(device)
    mkt_feat = torch.stack([r_t, tau_t, S0_t, lks_t], dim=1)
    X_cal    = torch.cat([cal_expanded, mkt_feat], dim=1)
    P_cal_norm = model_iv(X_cal)
    P_cal_abs  = (P_cal_norm * S0_t.view(-1, 1)).cpu().numpy().flatten()

P_mkt_arr = sample_df['P_mkt'].values

# ── BS IV 反解函数 ──────────────────────────────────────────────
def bs_call_price(S, K, r, tau, sigma):
    if sigma <= 0 or tau <= 0:
        return max(S - K * np.exp(-r * tau), 0.0)
    d1 = (np.log(S / K) + (r + 0.5 * sigma**2) * tau) / (sigma * np.sqrt(tau))
    d2 = d1 - sigma * np.sqrt(tau)
    return float(S * norm.cdf(d1) - K * np.exp(-r * tau) * norm.cdf(d2))

def implied_vol(price, S, K, r, tau, tol=1e-8):
    intrinsic = max(S - K * np.exp(-r * tau), 0.0)
    if price <= intrinsic + tol or tau <= 0:
        return np.nan
    try:
        f = lambda sig: bs_call_price(S, K, r, tau, sig) - price
        if f(1e-6) * f(10.0) > 0:
            return np.nan
        return brentq(f, 1e-6, 10.0, xtol=tol)
    except Exception:
        return np.nan

def compute_iv_array(prices, df):
    return np.array([
        implied_vol(float(prices[i]), float(df['S0'].iloc[i]),
                    float(df['K'].iloc[i]), float(df['r'].iloc[i]),
                    float(df['tau'].iloc[i]))
        for i in range(len(prices))
    ])

# ── 计算 IV ─────────────────────────────────────────────────────
print("正在计算真实价格的 IV...")
iv_true  = compute_iv_array(P_mkt_arr,      sample_df)
print("正在计算 Zhang 校准后价格的 IV...")
iv_zhang = compute_iv_array(P_cal_abs, sample_df)

# ── IV 空间误差统计 ─────────────────────────────────────────────
valid = np.isfinite(iv_true) & np.isfinite(iv_zhang) & (iv_true > 1e-8)
iv_rel_errs = np.abs(iv_zhang[valid] - iv_true[valid]) / (iv_true[valid] + 1e-8)
iv_mre     = iv_rel_errs.mean() * 100
iv_max_mre = iv_rel_errs.max()  * 100

# 价格空间 MRE（对比）
price_rel_errs = np.abs(P_cal_abs - P_mkt_arr) / (np.abs(P_mkt_arr) + 1e-8)
price_mre = price_rel_errs.mean() * 100

print()
print("=" * 62)
print("Zhang 方法 —— 合成数据 IV 空间误差分析")
print("=" * 62)
print(f"  有效样本数        : {valid.sum()} / {M}")
print(f"  IV MRE            : {iv_mre:.4f}%")
print(f"  IV 最大相对误差   : {iv_max_mre:.4f}%")
print()
print(f"  （对比）价格空间 MRE     : {price_mre:.4f}%")
print("=" * 62)


✅ 模型已加载 (device=mps)
✅ 合成数据期权数: 99
initial guess: [[ 3.18843882  0.27708885  0.13687617 -0.88595151  0.81513754]
 [ 4.56465033  0.61056942  0.7565469  -0.43791876  0.9357217 ]
 [ 4.08110924  0.01271112  0.87166385 -0.87145226  0.73235889]
 [ 0.88652155  0.86454713  0.5873151  -0.64524489  0.42846035]
 [ 0.15131516  0.13304044  0.70356197 -0.34988892  0.61923126]
 [ 1.924551    0.99723784  0.9827518  -0.31728931  0.65395468]
 [ 3.44534919  0.39503221  0.22158685 -0.28673491  0.53010078]
 [ 1.55810696  0.49097701  0.90053905 -0.10606301  0.36421724]
 [ 2.86193386  0.3286507   0.63487003 -0.61277546  0.39770281]
 [ 4.45246902  0.23488602  0.66086843 -0.82858696  0.83431771]]

✅ 校准完成，耗时 12.45s，loss=4.044668e-06
   校准参数: [ 3.5883794   0.08582912  0.7139488  -0.5438598   0.04003658]
正在计算真实价格的 IV...
正在计算 Zhang 校准后价格的 IV...

Zhang 方法 —— 合成数据 IV 空间误差分析
  有效样本数        : 99 / 99
  IV MRE            : 11.0397%
  IV 最大相对误差   : 62.8376%

  （对比）价格空间 MRE     : 184.7990%


---
## Part 2: 真实 SPY 期权数据 — 校准 & 跨日定价验证

**流程**:
1. 清洗 `spy_2022_09_01.csv` 和 `spy_2022_09_02.csv` 中的 Call 期权数据
2. 筛选条件: DTE ∈ [40, 212] 天, log(K/S₀) ∈ [-0.29, 0.43]
3. 分别用 9/1 和 9/2 数据校准 Heston 参数
4. 用两组参数分别通过 DDN 和 QuantLib 计算 9/2 的期权价格
5. 与 9/2 真实价格对比，计算 MRE = Σ|P_calc − P_real| / M

In [4]:
import numpy as np
import pandas as pd
from scipy.interpolate import interp1d

def interest_rate(date: str, tau: float) -> float: 
    """
    date: '09/01/2022' (确保 CSV 中的日期格式一致)
    tau: 剩余期限（年化，如 45/365）
    """
    # 期限节点 (年化): 1mo, 2mo, 3mo, 6mo, 1yr
    maturities = np.array([1/12, 2/12, 3/12, 6/12, 1.0]) 

    # 1. 读取数据 (假设 CSV 文件在当前目录下)
    data = pd.read_csv(ROOT / "data" / "par-yield-curve-rates-2020-2023.csv")
    
    # 2. 筛选特定日期并提取数值
    ir_row = data[data['date'] == date]
    if ir_row.empty:
        raise ValueError(f"未找到日期为 {date} 的数据，请检查日期格式是否为 YYYY-MM-DD 或 MM/DD/YYYY")

    # 关键修正：使用 .values.flatten() 将 DataFrame 的一行转为 1D 数组
    # 并且除以 100 将百分比转为小数 (例如 5.2 -> 0.052)
    market_rates = ir_row[['1 mo', '2 mo', '3 mo', '6 mo', '1 yr']].values.flatten() / 100.0

    # 步骤 1: 将市场单利转换为连续复利
    # 公式: r_c = ln(1 + R * t) / t 
    continuous_rates = np.log(1 + market_rates * maturities) / maturities

    # 步骤 2: 构建插值函数 (推荐在点数较少时先试用 'linear'，'cubic' 曲线更平滑但点少时易震荡)
    yield_curve = interp1d(maturities, continuous_rates, kind='cubic', fill_value="extrapolate")

    return float(yield_curve(tau))

# 3. 关键修正：if __name__ == '__main__': (去掉引号)
if __name__ == '__main__':
    # 假设期权链中有一个期权还有 45 天到期
    tau = 0.25
    try:
        exact_r = interest_rate('09/01/2022', tau)
        print(f"45天到期期权适用的连续复利无风险利率为: {exact_r:.6f}")
        # 输出示例: 0.024869 (即 2.4869%)
    except Exception as e:
        print(f"运行错误: {e}")

45天到期期权适用的连续复利无风险利率为: 0.029590


In [5]:
## Section 14: 清洗 SPY 真实期权 CSV 数据（与 Heston_Project 保持一致）

import pandas as pd
import numpy as np
from datetime import datetime
from pathlib import Path

DATA_DIR = Path.cwd() / "data"

# ── 筛选参数（与 Heston_Project/02_calibrate 保持一致） ──
DTE_LO, DTE_HI       = 40, 300          # 到期天数范围
LOG_KS_LO, LOG_KS_HI = -0.15, 0.15      # log-moneyness 范围（收紧到 ATM 附近）

def load_and_clean_spy(csv_path: str | Path, label: str = "") -> pd.DataFrame:
    """
    清洗 SPY 期权 CSV → 与 HestonCalibrator.calibrate 输入格式一致。

    输出列: r, tau, S0, K, P_mkt, log_K_S0
    仅保留 Call 期权（C_LAST > 0 且 C_VOLUME > 0 的行）。
    自适应从中间截取最多 10 条。
    """
    dt_obj = datetime.strptime(label, "%Y-%m-%d")
    date = dt_obj.strftime("%m/%d/%Y")

    maturities = np.array([1/12, 2/12, 3/12, 6/12, 1.0])
    yield_data = pd.read_csv(ROOT / "data" / "par-yield-curve-rates-2020-2023.csv")
    ir_row = yield_data[yield_data['date'] == date]
    if ir_row.empty:
        raise ValueError(f"未找到日期为 {date} 的数据")
    market_rates = ir_row[['1 mo', '2 mo', '3 mo', '6 mo', '1 yr']].values.flatten() / 100.0
    continuous_rates = np.log(1 + market_rates * maturities) / maturities
    yield_curve = interp1d(maturities, continuous_rates, kind='cubic', fill_value="extrapolate")

    raw = pd.read_csv(csv_path)
    raw.columns = [c.strip().strip("[]") for c in raw.columns]
    for col in ["UNDERLYING_LAST", "DTE", "STRIKE", "C_LAST", "C_BID", "C_ASK"]:
        raw[col] = pd.to_numeric(raw[col], errors="coerce")
    # ★ C_VOLUME: CSV 中可能为 str/空值，需先转数值
    raw["C_VOLUME"] = pd.to_numeric(raw["C_VOLUME"], errors="coerce")

    df = raw.dropna(subset=["UNDERLYING_LAST", "DTE", "STRIKE", "C_LAST"]).copy()
    df = df[df["C_LAST"] > 0.01].copy()
    df = df[df["UNDERLYING_LAST"] > 0].copy()
    df = df[df["C_VOLUME"].fillna(0) > 0].copy()   # ★ 确保有交易量
    df = df[(df["DTE"] >= DTE_LO) & (df["DTE"] <= DTE_HI)].copy()

    df["S0"]       = df["UNDERLYING_LAST"]
    df["K"]        = df["STRIKE"]
    df["P_mkt"]    = (df["C_BID"] + df["C_ASK"]) * 0.5
    df["tau"]      = df["DTE"] / 365.0
    df["r"]        = df["tau"].apply(lambda t: interest_rate(date, t))
    df["log_K_S0"] = np.log(df["K"] / df["S0"])

    df = df[(df["log_K_S0"] >= LOG_KS_LO) & (df["log_K_S0"] <= LOG_KS_HI)].copy()

    out = df[["r", "tau", "S0", "K", "P_mkt", "log_K_S0"]].reset_index(drop=True)

    # ── 自适应中间截取最多 100 条（防止越界） ──
    n_take = min(100, len(out))
    mid    = len(out) // 2
    start  = max(0, mid - n_take // 2)
    out    = out.iloc[start : start + n_take].reset_index(drop=True)

    print(f"[{label}] 清洗完成: {len(out)} 条有效 Call 期权")
    if len(out) == 0:
        raise ValueError(f"[{label}] 清洗后无有效数据，请检查筛选条件")
    print(f"  S0 = {out['S0'].iloc[0]:.2f},  DTE ∈ [{out['tau'].min()*365:.0f}, {out['tau'].max()*365:.0f}] 天")
    print(f"  K  ∈ [{out['K'].min():.1f}, {out['K'].max():.1f}]")
    print(f"  log(K/S0) ∈ [{out['log_K_S0'].min():.4f}, {out['log_K_S0'].max():.4f}]")
    print(f"  P_mkt ∈ [{out['P_mkt'].min():.2f}, {out['P_mkt'].max():.2f}]")
    return out

# ── 加载两天的数据 ──
df_sep1 = load_and_clean_spy(DATA_DIR / "spy_2022_09_01.csv", label="2022-09-01")
print("示例:")
display(df_sep1.head(5))
df_sep2 = load_and_clean_spy(DATA_DIR / "spy_2022_09_02.csv", label="2022-09-02")
print("示例:")
display(df_sep2.head(5))

[2022-09-01] 清洗完成: 100 条有效 Call 期权
  S0 = 396.39,  DTE ∈ [78, 106] 天
  K  ∈ [350.0, 460.0]
  log(K/S0) ∈ [-0.1245, 0.1488]
  P_mkt ∈ [0.49, 54.30]
示例:


,r,tau,S0,K,P_mkt,log_K_S0
0,0.028941,0.213808,396.39,392.0,19.730,-0.011137
1,0.028941,0.213808,396.39,394.0,18.485,-0.006048
2,0.028941,0.213808,396.39,395.0,17.900,-0.003513
3,0.028941,0.213808,396.39,396.0,17.420,-0.000984
4,0.028941,0.213808,396.39,398.0,16.305,0.004053


[2022-09-02] 清洗完成: 100 条有效 Call 期权
  S0 = 392.27,  DTE ∈ [77, 105] 天
  K  ∈ [340.0, 455.0]
  log(K/S0) ∈ [-0.1430, 0.1483]
  P_mkt ∈ [0.52, 58.33]
示例:


,r,tau,S0,K,P_mkt,log_K_S0
0,0.028698,0.211068,392.27,380.0,24.500,-0.031779
1,0.028698,0.211068,392.27,385.0,21.275,-0.018707
2,0.028698,0.211068,392.27,386.0,20.630,-0.016113
3,0.028698,0.211068,392.27,388.0,19.430,-0.010945
4,0.028698,0.211068,392.27,390.0,18.195,-0.005804


In [6]:
## Section 15: 分别用 9/1 和 9/2 数据校准 Heston 参数

from modules.calibration import HestonCalibrator
import time
calibrator = HestonCalibrator(model, device, seed=42)

def build_market_dict(df: pd.DataFrame) -> dict:
    return {
        "r":     df["r"].values.astype(np.float32),
        "tau":   df["tau"].values.astype(np.float32),
        "S0":    df["S0"].values.astype(np.float32),
        "K":     df["K"].values.astype(np.float32),
        "P_mkt": df["P_mkt"].values.astype(np.float32),
    }

# ── 9月1日 校准 ──
t0 = time.time()
print("=" * 60)
print("校准 — 2022-09-01 SPY 期权数据")
print("=" * 60)
theta_sep1, loss_sep1, _ = calibrator.calibrate(
    build_market_dict(df_sep1),
    n_starts=10, lr=5e-3, max_steps=300, patience=50, verbose=True,
)
T0 = time.time() - t0

# ── 9月2日 校准 ──
t1 = time.time()
print("\n" + "=" * 60)
print("校准 — 2022-09-02 SPY 期权数据")
print("=" * 60)
theta_sep2, loss_sep2, _ = calibrator.calibrate(
    build_market_dict(df_sep2),
    n_starts=10, lr=5e-3, max_steps=300, patience=50, verbose=True,
)
T1 = time.time() - t1

# ── 对比两天校准结果 ──
param_names = ["kappa", "lambda", "sigma", "rho", "v0"]
print("\n" + "=" * 60)
print("两天校准参数对比")
print("=" * 60)
print(f"  {'参数':>8s} | {'9/1 校准值':>12s} | {'9/2 校准值':>12s} | {'变化量':>10s}")
print("  " + "-" * 52)
for n, v1, v2 in zip(param_names, theta_sep1, theta_sep2):
    diff = v2 - v1
    print(f"  {n:>8s} | {v1:12.6f} | {v2:12.6f} | {diff:+10.6f}")
print(f"\n  9/1 loss(norm) = {loss_sep1:.6e}")
print(f"  9/2 loss(norm) = {loss_sep2:.6e}")
print(f"9/1 校准耗时: {T0} 秒")
print(f"9/2 校准耗时: {T1} 秒")

校准 — 2022-09-01 SPY 期权数据
initial guess: [[ 3.87204068  0.44448966  0.87273813 -0.30723718  0.10323557]
 [ 4.87835553  0.7635283   0.80745787 -0.79110341  0.45588208]
 [ 1.86028214  0.92749734  0.67947861 -0.20065263  0.44898006]
 [ 1.14392122  0.55903894  0.15743553 -0.1965135   0.63534776]
 [ 3.79285782  0.36098071  0.97362822 -0.14084705  0.78059966]
 [ 0.98124715  0.47205379  0.13942339 -0.76885393  0.68621846]
 [ 3.72636316  0.96783464  0.39324282 -0.58510925  0.47486025]
 [ 0.95546208  0.13862229  0.52813443 -0.70712705  0.67311585]
 [ 2.19138808  0.83435141  0.73023859 -0.63448835  0.8339372 ]
 [ 4.02577414  0.3936036   0.35949529 -0.31987882  0.14835496]]

  市场期权数   : 100
  初始点数     : 10
  Adam lr      : 0.005，max_steps=300，patience=50
  price_norm   : [0.0012, 0.1370]

  [start  1/10] step= 100 | loss(norm)=1.5885e-06 | lr=5.00e-03
  [start  1/10] step= 200 | loss(norm)=1.4350e-06 | lr=5.00e-03
  [start  1/10] step= 300 | loss(norm)=1.3577e-06 | lr=5.00e-03
  [start  1/10] loss

In [7]:
## Section 16: 用校准参数对 9/2 数据定价 — DDN vs QuantLib vs 真实价格

from modules.pricing import calculate_heston_price
import time
M = len(df_sep2)
P_real = df_sep2["P_mkt"].values  # 9/2 真实 Call 价格

# ── 辅助: DDN 定价 ──────────────────────────────────────────────
def ddn_price_batch(theta_np: np.ndarray, df: pd.DataFrame) -> np.ndarray:
    """用 DDN 模型对整批期权定价，返回绝对价格数组。"""
    n = len(df)
    model.eval()
    with torch.no_grad():
        th   = torch.tensor(theta_np, dtype=torch.float32, device=device)
        th_e = th.unsqueeze(0).expand(n, -1)                     # (n, 5)
        S0_t = torch.tensor(df["S0"].values, dtype=torch.float32, device=device)
        r_t  = torch.tensor(df["r"].values,  dtype=torch.float32, device=device)
        tau_t = torch.tensor(df["tau"].values, dtype=torch.float32, device=device)
        lks_t = torch.tensor(df["log_K_S0"].values, dtype=torch.float32, device=device)
        mkt   = torch.stack([r_t, tau_t, S0_t, lks_t], dim=1)    # (n, 4)
        X     = torch.cat([th_e, mkt], dim=1)                     # (n, 9)
        P_norm = model(X)                                          # price_norm
        P_abs  = (P_norm * S0_t.view(-1, 1)).cpu().numpy().flatten()
    return P_abs

# ── 辅助: QuantLib 定价 ────────────────────────────────────────
def ql_price_batch(theta_np: np.ndarray, df: pd.DataFrame) -> np.ndarray:
    """用 QuantLib 逐条定价，返回绝对价格数组。"""
    kappa, lam, sigma, rho, v0 = theta_np
    prices = []
    for _, row in df.iterrows():
        p = calculate_heston_price(
            float(kappa), float(lam), float(sigma), float(rho), float(v0),
            float(row["r"]), float(row["tau"]), float(row["S0"]), float(row["K"]),
        )
        prices.append(p if not np.isnan(p) else 0.0)
    return np.array(prices)

# ── 辅助: MRE 计算 ─────────────────────────────────────────────
def calc_mre(P_calc: np.ndarray, P_real: np.ndarray) -> float:
    """MRE = Σ|P_calc - P_real| / M"""
    return float(np.sum(np.abs(P_calc - P_real) / P_real) / len(P_real))

# ── 4 组定价 ───────────────────────────────────────────────────
print("正在计算 DDN 定价...")
P_ddn_from_sep1 = ddn_price_batch(theta_sep1, df_sep2)   # 9/1 参数 + DDN
P_ddn_from_sep2 = ddn_price_batch(theta_sep2, df_sep2)   # 9/2 参数 + DDN

print("正在计算 QuantLib 定价（较慢）...")
P_ql_from_sep1  = ql_price_batch(theta_sep1, df_sep2)    # 9/1 参数 + QuantLib
P_ql_from_sep2  = ql_price_batch(theta_sep2, df_sep2)    # 9/2 参数 + QuantLib

# ── MRE 汇总 ──────────────────────────────────────────────────
mre_ddn_sep1 = calc_mre(P_ddn_from_sep1, P_real)
mre_ddn_sep2 = calc_mre(P_ddn_from_sep2, P_real)
mre_ql_sep1  = calc_mre(P_ql_from_sep1,  P_real)
mre_ql_sep2  = calc_mre(P_ql_from_sep2,  P_real)

print("\n" + "=" * 65)
print(f"跨日定价验证 — 9/2 真实 Call 期权 ({M} 条)")
print("=" * 65)
print(f"  {'方法':>20s} | {'校准日':>8s} | {'MRE (Σ|err|/M)':>16s}")
print("  " + "-" * 52)
print(f"  {'DDN':>20s} | {'9/1':>8s} | {mre_ddn_sep1:16.4f}")
print(f"  {'DDN':>20s} | {'9/2':>8s} | {mre_ddn_sep2:16.4f}")
print(f"  {'QuantLib':>20s} | {'9/1':>8s} | {mre_ql_sep1:16.4f}")
print(f"  {'QuantLib':>20s} | {'9/2':>8s} | {mre_ql_sep2:16.4f}")
print()
print(f"  💡 DDN 与 QuantLib 之差 (9/1参数): {abs(mre_ddn_sep1 - mre_ql_sep1):.4f}")
print(f"  💡 DDN 与 QuantLib 之差 (9/2参数): {abs(mre_ddn_sep2 - mre_ql_sep2):.4f}")

正在计算 DDN 定价...
正在计算 QuantLib 定价（较慢）...

跨日定价验证 — 9/2 真实 Call 期权 (100 条)
                    方法 |      校准日 |   MRE (Σ|err|/M)
  ----------------------------------------------------
                   DDN |      9/1 |           0.1097
                   DDN |      9/2 |           0.1336
              QuantLib |      9/1 |           0.2070
              QuantLib |      9/2 |           0.1693

  💡 DDN 与 QuantLib 之差 (9/1参数): 0.0973
  💡 DDN 与 QuantLib 之差 (9/2参数): 0.0356


In [8]:
## Section 17: 逐条定价明细 & 误差分布

# ── 9/1 参数 → 9/2 定价明细（DDN）──
rel_err_ddn1 = np.abs(P_ddn_from_sep1 - P_real) / (np.abs(P_real) + 1e-8) * 100
rel_err_ql1  = np.abs(P_ql_from_sep1 - P_real)  / (np.abs(P_real) + 1e-8) * 100

print("=" * 80)
print(f"9/1 校准参数 → 9/2 定价明细（前 20 条）")
print("=" * 80)
print(f"  {'#':>4s} | {'K':>8s} | {'tau':>6s} | {'P_real':>9s} | "
      f"{'DDN':>9s} | {'QL':>9s} | {'DDN%':>7s} | {'QL%':>7s}")
print("  " + "-" * 72)
for i in range(min(20, M)):
    flag = "✅" if rel_err_ddn1[i] < 5.0 else "❌"
    print(f"  {flag}{i+1:3d} | {df_sep2['K'].iloc[i]:8.1f} | "
          f"{df_sep2['tau'].iloc[i]:6.3f} | {P_real[i]:9.2f} | "
          f"{P_ddn_from_sep1[i]:9.2f} | {P_ql_from_sep1[i]:9.2f} | "
          f"{rel_err_ddn1[i]:6.2f}% | {rel_err_ql1[i]:6.2f}%")

# ── 误差统计 ──
print(f"\n{'─'*65}")
print(f"误差分布统计（9/1 参数 → 9/2 定价，{M} 条）:")
print(f"{'─'*65}")
for name, errs in [("DDN", rel_err_ddn1), ("QuantLib", rel_err_ql1)]:
    print(f"\n  [{name}]")
    print(f"    均值相对误差   : {errs.mean():.4f}%")
    print(f"    中位数相对误差 : {np.median(errs):.4f}%")
    print(f"    最大相对误差   : {errs.max():.4f}%")
    print(f"    < 1%  占比     : {(errs < 1.0).mean()*100:.1f}%")
    print(f"    < 5%  占比     : {(errs < 5.0).mean()*100:.1f}%")
    print(f"    < 10% 占比     : {(errs < 10.0).mean()*100:.1f}%")

# ── 同样对 9/2 参数做统计 ──
rel_err_ddn2 = np.abs(P_ddn_from_sep2 - P_real) / (np.abs(P_real) + 1e-8) * 100
rel_err_ql2  = np.abs(P_ql_from_sep2 - P_real)  / (np.abs(P_real) + 1e-8) * 100
print("=" * 80)
print(f"9/2 校准参数 → 9/2 定价明细（前 20 条）")
print("=" * 80)
print(f"  {'#':>4s} | {'K':>8s} | {'tau':>6s} | {'P_real':>9s} | "
      f"{'DDN':>9s} | {'QL':>9s} | {'DDN%':>7s} | {'QL%':>7s}")
print("  " + "-" * 72)
for i in range(min(20, M)):
    flag = "✅" if rel_err_ddn2[i] < 5.0 else "❌"
    print(f"  {flag}{i+1:3d} | {df_sep2['K'].iloc[i]:8.1f} | "
          f"{df_sep2['tau'].iloc[i]:6.3f} | {P_real[i]:9.2f} | "
          f"{P_ddn_from_sep2[i]:9.2f} | {P_ql_from_sep2[i]:9.2f} | "
          f"{rel_err_ddn2[i]:6.2f}% | {rel_err_ql2[i]:6.2f}%")
    
print(f"\n{'─'*65}")
print(f"误差分布统计（9/2 参数 → 9/2 定价，{M} 条）:")
print(f"{'─'*65}")
for name, errs in [("DDN", rel_err_ddn2), ("QuantLib", rel_err_ql2)]:
    print(f"\n  [{name}]")
    print(f"    均值相对误差   : {errs.mean():.4f}%")
    print(f"    中位数相对误差 : {np.median(errs):.4f}%")
    print(f"    最大相对误差   : {errs.max():.4f}%")
    print(f"    < 1%  占比     : {(errs < 1.0).mean()*100:.1f}%")
    print(f"    < 5%  占比     : {(errs < 5.0).mean()*100:.1f}%")
    print(f"    < 10% 占比     : {(errs < 10.0).mean()*100:.1f}%")

print(f"\n✅ 真实 SPY 数据校准 & 跨日定价验证完成！")

9/1 校准参数 → 9/2 定价明细（前 20 条）
     # |        K |    tau |    P_real |       DDN |        QL |    DDN% |     QL%
  ------------------------------------------------------------------------
  ✅  1 |    380.0 |  0.211 |     24.50 |     24.42 |     22.57 |   0.33% |   7.88%
  ✅  2 |    385.0 |  0.211 |     21.27 |     20.94 |     19.15 |   1.56% |   9.97%
  ✅  3 |    386.0 |  0.211 |     20.63 |     20.28 |     18.50 |   1.71% |  10.32%
  ✅  4 |    388.0 |  0.211 |     19.43 |     18.97 |     17.22 |   2.35% |  11.35%
  ✅  5 |    390.0 |  0.211 |     18.20 |     17.71 |     15.99 |   2.66% |  12.12%
  ✅  6 |    392.0 |  0.211 |     17.04 |     16.49 |     14.80 |   3.21% |  13.15%
  ✅  7 |    394.0 |  0.211 |     15.95 |     15.32 |     13.65 |   3.95% |  14.40%
  ✅  8 |    395.0 |  0.211 |     15.35 |     14.75 |     13.10 |   3.90% |  14.67%
  ✅  9 |    396.0 |  0.211 |     14.82 |     14.19 |     12.56 |   4.22% |  15.28%
  ✅ 10 |    398.0 |  0.211 |     13.77 |     13.12 |     11.51 |   


---
## Part 3: NVDA 真实期权数据 — 2021-11-01 / 2021-11-02 当日与跨日校准

该新增部分只使用 NVDA 数据，不依赖前面 SPY cell 的运行状态；会重新加载已保存模型权重、清洗两天的 NVDA call option、按 maturity/strike 选取 100 条合约、匹配利率曲线，并同时运行 `Heston_Project` 与 `Zhang_realize` 两个模型，最后在 `experiments result/` 下输出 CSV/JSON 和 LaTeX 表格。


In [ ]:

# NVDA 2021-11-01 / 2021-11-02 calibration block
# Self-contained: do not rerun earlier SPY cells.

from __future__ import annotations

import os
os.environ.setdefault("NUMBA_DISABLE_JIT", "1")

import sys
import json
import math
import time
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import torch
from scipy.interpolate import interp1d
import py_vollib_vectorized as pvv

PROJECT_DIR = Path.cwd().resolve()
ROOT = PROJECT_DIR.parent if PROJECT_DIR.name in {"Heston_Project", "Zhang_realize"} else PROJECT_DIR
NVDA_CSV = ROOT / "data" / "nvda_2020_2022.csv"
OUT_DIR = ROOT / "experiments result"
OUT_DIR.mkdir(exist_ok=True)

DATES = ("2021-11-01", "2021-11-02")
N_CONTRACTS = 100
DTE_LO, DTE_HI = 40, 300
LOG_KS_LO, LOG_KS_HI = -0.15, 0.20


def pct_tex(x: float | None) -> str:
    if x is None or (isinstance(x, float) and (math.isnan(x) or math.isinf(x))):
        return "n/a"
    return f"{100.0 * float(x):.2f}\\%"


def pct_text(x: float | None) -> str:
    if x is None or (isinstance(x, float) and (math.isnan(x) or math.isinf(x))):
        return "n/a"
    return f"{100.0 * float(x):.2f}%"


def matched_interest_rate(yield_data: pd.DataFrame, quote_date: str, tau: float) -> float:
    date = pd.to_datetime(quote_date).strftime("%m/%d/%Y")
    maturities = np.array([1 / 12, 2 / 12, 3 / 12, 6 / 12, 1.0], dtype=float)
    row = yield_data[yield_data["date"] == date]
    if row.empty:
        raise ValueError(f"No yield curve row for {date}")
    par_rates = row[["1 mo", "2 mo", "3 mo", "6 mo", "1 yr"]].values.flatten().astype(float) / 100.0
    continuous_rates = np.log(1.0 + par_rates * maturities) / maturities
    curve = interp1d(maturities, continuous_rates, kind="cubic", fill_value="extrapolate")
    return float(curve(tau))


def compute_iv_vega(price, s0, strike, tau, rate):
    price_s = pd.Series(np.asarray(price, dtype=float))
    s0_s = pd.Series(np.asarray(s0, dtype=float))
    strike_s = pd.Series(np.asarray(strike, dtype=float))
    tau_s = pd.Series(np.asarray(tau, dtype=float))
    rate_s = pd.Series(np.asarray(rate, dtype=float))
    flag_s = pd.Series(["c"] * len(price_s))
    iv_s = pvv.vectorized_implied_volatility(
        price_s,
        s0_s,
        strike_s,
        tau_s,
        rate_s,
        flag_s,
        q=0.0,
        model="black_scholes_merton",
        return_as="series",
    )
    vega_s = pvv.vectorized_vega(
        flag_s,
        s0_s,
        strike_s,
        tau_s,
        rate_s,
        iv_s,
        q=0.0,
        model="black_scholes_merton",
        return_as="series",
    )
    return iv_s.values.astype(float), vega_s.values.astype(float)


def load_nvda_rows() -> pd.DataFrame:
    needed = {"QUOTE_DATE", "UNDERLYING_LAST", "DTE", "STRIKE", "C_LAST", "C_BID", "C_ASK", "C_VOLUME"}
    pieces = []
    for chunk in pd.read_csv(
        NVDA_CSV,
        chunksize=200_000,
        dtype=str,
        usecols=lambda c: c.strip().strip("[]") in needed,
    ):
        chunk.columns = [c.strip().strip("[]") for c in chunk.columns]
        chunk["QUOTE_DATE"] = chunk["QUOTE_DATE"].astype(str).str.strip()
        mask = chunk["QUOTE_DATE"].isin(DATES)
        if mask.any():
            pieces.append(chunk.loc[mask].copy())
    if not pieces:
        raise ValueError("No NVDA rows found for 2021-11-01 and 2021-11-02")
    return pd.concat(pieces, ignore_index=True)


_RAW_NVDA = load_nvda_rows()


def _allocate_even_maturity_counts(df: pd.DataFrame, total: int) -> dict[float, int]:
    """Split the target sample size as evenly as possible across available maturities."""
    dtes = sorted(df["DTE"].unique())
    if not dtes:
        return {}
    capacity = df.groupby("DTE").size().to_dict()
    total = min(total, int(sum(capacity.values())))

    base = total // len(dtes)
    remainder = total % len(dtes)
    counts = {dte: min(int(capacity[dte]), base + (i < remainder)) for i, dte in enumerate(dtes)}

    # If a maturity bucket has fewer rows than its target, recycle the spare slots
    # round-robin into other maturities that still have available strikes.
    while sum(counts.values()) < total:
        changed = False
        for dte in dtes:
            if counts[dte] < int(capacity[dte]) and sum(counts.values()) < total:
                counts[dte] += 1
                changed = True
        if not changed:
            break
    return counts


def _centered_strike_slice(group: pd.DataFrame, n_take: int) -> pd.DataFrame:
    """For one maturity, take a contiguous strike window centered around the spot price."""
    group = group.sort_values("K").reset_index(drop=True)
    if n_take >= len(group):
        return group

    strikes = group["K"].values.astype(float)
    s0 = float(group["S0"].median())
    center = int(np.searchsorted(strikes, s0, side="left"))
    start = max(0, center - n_take // 2)
    end = start + n_take
    if end > len(group):
        end = len(group)
        start = max(0, end - n_take)
    return group.iloc[start:end]


def select_uniform_atm_nvda_contracts(df: pd.DataFrame, n_contracts: int = N_CONTRACTS) -> pd.DataFrame:
    """
    Select NVDA contracts in the same spirit as the SPY market-data block:
    keep liquid/valid calls first, then use a deterministic slice around the
    underlying price. The sample is balanced across maturities, and each
    maturity contributes a centered, nearly uniform strike window around S0.
    """
    counts = _allocate_even_maturity_counts(df, n_contracts)
    pieces = []
    for dte, n_take in counts.items():
        maturity_slice = df[df["DTE"] == dte].copy()
        pieces.append(_centered_strike_slice(maturity_slice, n_take))

    selected = pd.concat(pieces, ignore_index=True)
    selected = selected.sort_values(["DTE", "K"]).reset_index(drop=True)
    if len(selected) < min(n_contracts, len(df)):
        raise ValueError(f"Only selected {len(selected)} contracts out of requested {n_contracts}")
    return selected


def load_and_clean_nvda(yield_data: pd.DataFrame, label: str) -> pd.DataFrame:
    df = _RAW_NVDA[_RAW_NVDA["QUOTE_DATE"] == label].copy()
    for col in ["UNDERLYING_LAST", "DTE", "STRIKE", "C_LAST", "C_BID", "C_ASK", "C_VOLUME"]:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    # Basic quote-quality filters: valid numeric call quotes, positive volume,
    # maturities in the same broad range used by the SPY experiment.
    df = df.dropna(subset=["UNDERLYING_LAST", "DTE", "STRIKE", "C_LAST", "C_BID", "C_ASK"]).copy()
    df = df[df["C_LAST"] > 0.01].copy()
    df = df[df["UNDERLYING_LAST"] > 0].copy()
    df = df[df["C_VOLUME"].fillna(0) > 0].copy()
    df = df[(df["DTE"] >= DTE_LO) & (df["DTE"] <= DTE_HI)].copy()

    df["S0"] = df["UNDERLYING_LAST"]
    df["K"] = df["STRIKE"]
    df["P_mkt"] = (df["C_BID"] + df["C_ASK"]) * 0.5
    df["tau"] = df["DTE"] / 365.0
    df["log_K_S0"] = np.log(df["K"] / df["S0"])
    df = df[(df["log_K_S0"] >= LOG_KS_LO) & (df["log_K_S0"] <= LOG_KS_HI)].copy()
    df = df[df["P_mkt"] > 0].copy()

    # Match each contract's exact maturity to the daily Treasury par-yield curve.
    df["r"] = df["tau"].map(lambda t: matched_interest_rate(yield_data, label, float(t)))

    # Convert mid quotes to Black-Scholes IV/Vega and remove inversion failures.
    iv_arr, vega_arr = compute_iv_vega(df["P_mkt"].values, df["S0"].values, df["K"].values, df["tau"].values, df["r"].values)
    df["iv_market"] = iv_arr
    df["vega"] = vega_arr
    df = df.dropna(subset=["iv_market", "vega"]).copy()
    df = df[(df["iv_market"] > 0) & (df["vega"] > 0)].copy()

    df["rel_spread"] = (df["C_ASK"] - df["C_BID"]) / df["P_mkt"].replace(0, np.nan)
    out = select_uniform_atm_nvda_contracts(df, N_CONTRACTS)

    cols = ["r", "tau", "S0", "K", "P_mkt", "log_K_S0", "iv_market", "vega", "DTE", "C_VOLUME", "rel_spread"]
    out = out[cols].copy()
    maturity_counts = out.groupby("DTE").size()
    print(
        f"[{label}] selected {len(out)} NVDA calls across {maturity_counts.size} maturities: "
        f"S0={out['S0'].iloc[0]:.2f}, "
        f"K=[{out['K'].min():.2f}, {out['K'].max():.2f}], "
        f"DTE=[{out['DTE'].min():.0f}, {out['DTE'].max():.0f}], "
        f"contracts/maturity=[{maturity_counts.min()}, {maturity_counts.max()}], "
        f"IV=[{out['iv_market'].min():.4f}, {out['iv_market'].max():.4f}]"
    )
    return out

def clear_project_modules():
    for name in list(sys.modules):
        if name == "modules" or name.startswith("modules."):
            del sys.modules[name]


def load_project_model(project_root: Path, default_input_dim: int) -> dict:
    clear_project_modules()
    sys.path.insert(0, str(project_root))
    cwd = Path.cwd()
    os.chdir(project_root)
    try:
        from modules.model import HestonDDN
        from modules.calibration import HestonCalibrator
        from modules.pricing import calculate_heston_price
        try:
            from modules.pricing import check_feller
        except Exception:
            check_feller = lambda kappa, theta, sigma: 2.0 * kappa * theta > sigma**2

        model_path = project_root / "models" / "heston_ddn_weights.pth"
        device = torch.device("mps" if torch.backends.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu")
        ckpt = torch.load(model_path, map_location=device)
        model = HestonDDN(input_dim=ckpt.get("input_dim", default_input_dim), heston_param_dim=ckpt.get("heston_dim", 5)).to(device)
        model.load_state_dict(ckpt["model_state_dict"])
        model.is_fitted = ckpt.get("is_fitted", True)
        model.eval()
        return {
            "device": device,
            "model": model,
            "calibrator_cls": HestonCalibrator,
            "calculate_heston_price": calculate_heston_price,
            "check_feller": check_feller,
        }
    finally:
        os.chdir(cwd)
        try:
            sys.path.remove(str(project_root))
        except ValueError:
            pass


def market_dict_heston(df: pd.DataFrame) -> dict:
    return {
        "r": df["r"].values.astype(np.float32),
        "tau": df["tau"].values.astype(np.float32),
        "S0": df["S0"].values.astype(np.float32),
        "K": df["K"].values.astype(np.float32),
        "iv_market": df["iv_market"].values.astype(np.float64),
        "vega": df["vega"].values.astype(np.float64),
    }


def market_dict_zhang(df: pd.DataFrame) -> dict:
    return {
        "r": df["r"].values.astype(np.float32),
        "tau": df["tau"].values.astype(np.float32),
        "S0": df["S0"].values.astype(np.float32),
        "K": df["K"].values.astype(np.float32),
        "P_mkt": df["P_mkt"].values.astype(np.float32),
    }


def calc_mre(pred, true) -> float:
    pred = np.asarray(pred, dtype=float)
    true = np.asarray(true, dtype=float)
    valid = np.isfinite(pred) & np.isfinite(true) & (np.abs(true) > 1e-12)
    if not valid.any():
        return float("nan")
    return float(np.mean(np.abs(pred[valid] - true[valid]) / np.abs(true[valid])))


def error_stats(pred, true) -> dict:
    pred = np.asarray(pred, dtype=float)
    true = np.asarray(true, dtype=float)
    valid = np.isfinite(pred) & np.isfinite(true) & (np.abs(true) > 1e-12)
    err = np.full_like(true, np.nan, dtype=float)
    err[valid] = np.abs(pred[valid] - true[valid]) / np.abs(true[valid])
    return {
        "mre": float(np.nanmean(err)),
        "median": float(np.nanmedian(err)),
        "max": float(np.nanmax(err)),
        "lt_1pct": float(np.nanmean(err < 0.01)),
        "lt_5pct": float(np.nanmean(err < 0.05)),
        "lt_10pct": float(np.nanmean(err < 0.10)),
        "valid": int(valid.sum()),
        "total": int(len(true)),
    }


def ql_price_batch(calc_price, theta, df: pd.DataFrame) -> np.ndarray:
    kappa, theta_bar, sigma, rho, v0 = [float(x) for x in theta]
    prices = []
    for _, row in df.iterrows():
        p = calc_price(kappa, theta_bar, sigma, rho, v0, float(row["r"]), float(row["tau"]), float(row["S0"]), float(row["K"]))
        prices.append(p if np.isfinite(p) else np.nan)
    return np.asarray(prices, dtype=float)


def iv_from_prices(prices, df: pd.DataFrame) -> np.ndarray:
    iv_arr, _ = compute_iv_vega(prices, df["S0"].values, df["K"].values, df["tau"].values, df["r"].values)
    return iv_arr


def ddn_iv_batch(model, device, theta, df: pd.DataFrame) -> np.ndarray:
    n = len(df)
    model.eval()
    with torch.no_grad():
        theta_t = torch.tensor(theta, dtype=torch.float32, device=device).unsqueeze(0).expand(n, -1)
        r_t = torch.tensor(df["r"].values, dtype=torch.float32, device=device)
        tau_t = torch.tensor(df["tau"].values, dtype=torch.float32, device=device)
        lks_t = torch.tensor(df["log_K_S0"].values, dtype=torch.float32, device=device)
        x = torch.cat([theta_t, torch.stack([r_t, tau_t, lks_t], dim=1)], dim=1)
        return model(x).cpu().numpy().flatten()


def ddn_price_batch(model, device, theta, df: pd.DataFrame) -> np.ndarray:
    n = len(df)
    model.eval()
    with torch.no_grad():
        theta_t = torch.tensor(theta, dtype=torch.float32, device=device).unsqueeze(0).expand(n, -1)
        s0_t = torch.tensor(df["S0"].values, dtype=torch.float32, device=device)
        r_t = torch.tensor(df["r"].values, dtype=torch.float32, device=device)
        tau_t = torch.tensor(df["tau"].values, dtype=torch.float32, device=device)
        lks_t = torch.tensor(df["log_K_S0"].values, dtype=torch.float32, device=device)
        x = torch.cat([theta_t, torch.stack([r_t, tau_t, s0_t, lks_t], dim=1)], dim=1)
        p_norm = model(x).cpu().numpy().flatten()
    return p_norm * df["S0"].values.astype(float)


def run_heston_project(df_1101: pd.DataFrame, df_1102: pd.DataFrame) -> dict:
    project = load_project_model(ROOT / "Heston_Project", default_input_dim=8)
    model, device = project["model"], project["device"]
    calibrator = project["calibrator_cls"](model, device, seed=42)

    t0 = time.time()
    theta_1101, loss_1101, _ = calibrator.calibrate(
        market_dict_heston(df_1101),
        n_starts=10,
        lr=5e-3,
        max_steps=300,
        patience=50,
        lambda_feller=10.0,
        verbose=False,
    )
    time_1101 = time.time() - t0

    t0 = time.time()
    theta_1102, loss_1102, _ = calibrator.calibrate(
        market_dict_heston(df_1102),
        n_starts=10,
        lr=5e-3,
        max_steps=300,
        patience=50,
        lambda_feller=10.0,
        verbose=False,
    )
    time_1102 = time.time() - t0

    iv_true = df_1102["iv_market"].values
    price_true = df_1102["P_mkt"].values
    calc_price = project["calculate_heston_price"]
    scenarios = {}
    for scenario, theta in [("cross_day", theta_1101), ("same_day", theta_1102)]:
        ddn_iv = ddn_iv_batch(model, device, theta, df_1102)
        ql_price = ql_price_batch(calc_price, theta, df_1102)
        ql_iv = iv_from_prices(ql_price, df_1102)
        scenarios[scenario] = {
            "ddn_iv_mre": calc_mre(ddn_iv, iv_true),
            "ql_iv_mre": calc_mre(ql_iv, iv_true),
            "ql_price_mre": calc_mre(ql_price, price_true),
            "ddn_iv_stats": error_stats(ddn_iv, iv_true),
            "ql_iv_stats": error_stats(ql_iv, iv_true),
            "ql_price_stats": error_stats(ql_price, price_true),
            "ddn_iv": [float(x) for x in ddn_iv],
            "ql_iv": [float(x) for x in ql_iv],
            "ql_price": [float(x) for x in ql_price],
        }

    return {
        "model": "Heston_Project",
        "objective": "Vega-weighted IV MSE + Feller penalty",
        "theta_2021_11_01": [float(x) for x in theta_1101],
        "theta_2021_11_02": [float(x) for x in theta_1102],
        "loss_2021_11_01": float(loss_1101),
        "loss_2021_11_02": float(loss_1102),
        "time_2021_11_01_sec": float(time_1101),
        "time_2021_11_02_sec": float(time_1102),
        "feller_2021_11_01": bool(project["check_feller"](float(theta_1101[0]), float(theta_1101[1]), float(theta_1101[2]))),
        "feller_2021_11_02": bool(project["check_feller"](float(theta_1102[0]), float(theta_1102[1]), float(theta_1102[2]))),
        "scenarios": scenarios,
    }


def run_zhang_realize(df_1101: pd.DataFrame, df_1102: pd.DataFrame) -> dict:
    project = load_project_model(ROOT / "Zhang_realize", default_input_dim=9)
    model, device = project["model"], project["device"]
    calibrator = project["calibrator_cls"](model, device, seed=42)

    t0 = time.time()
    theta_1101, loss_1101, _ = calibrator.calibrate(
        market_dict_zhang(df_1101),
        n_starts=10,
        lr=5e-3,
        max_steps=300,
        patience=50,
        verbose=False,
    )
    time_1101 = time.time() - t0

    t0 = time.time()
    theta_1102, loss_1102, _ = calibrator.calibrate(
        market_dict_zhang(df_1102),
        n_starts=10,
        lr=5e-3,
        max_steps=300,
        patience=50,
        verbose=False,
    )
    time_1102 = time.time() - t0

    iv_true = df_1102["iv_market"].values
    price_true = df_1102["P_mkt"].values
    calc_price = project["calculate_heston_price"]
    scenarios = {}
    for scenario, theta in [("cross_day", theta_1101), ("same_day", theta_1102)]:
        ddn_price = ddn_price_batch(model, device, theta, df_1102)
        ddn_iv = iv_from_prices(ddn_price, df_1102)
        ql_price = ql_price_batch(calc_price, theta, df_1102)
        ql_iv = iv_from_prices(ql_price, df_1102)
        scenarios[scenario] = {
            "ddn_price_mre": calc_mre(ddn_price, price_true),
            "ddn_iv_mre": calc_mre(ddn_iv, iv_true),
            "ql_price_mre": calc_mre(ql_price, price_true),
            "ql_iv_mre": calc_mre(ql_iv, iv_true),
            "ddn_price_stats": error_stats(ddn_price, price_true),
            "ddn_iv_stats": error_stats(ddn_iv, iv_true),
            "ql_price_stats": error_stats(ql_price, price_true),
            "ql_iv_stats": error_stats(ql_iv, iv_true),
            "ddn_price": [float(x) for x in ddn_price],
            "ddn_iv": [float(x) for x in ddn_iv],
            "ql_price": [float(x) for x in ql_price],
            "ql_iv": [float(x) for x in ql_iv],
        }

    return {
        "model": "Zhang_realize",
        "objective": "Normalised price MSE",
        "theta_2021_11_01": [float(x) for x in theta_1101],
        "theta_2021_11_02": [float(x) for x in theta_1102],
        "loss_2021_11_01": float(loss_1101),
        "loss_2021_11_02": float(loss_1102),
        "time_2021_11_01_sec": float(time_1101),
        "time_2021_11_02_sec": float(time_1102),
        "feller_2021_11_01": bool(2.0 * float(theta_1101[0]) * float(theta_1101[1]) > float(theta_1101[2]) ** 2),
        "feller_2021_11_02": bool(2.0 * float(theta_1102[0]) * float(theta_1102[1]) > float(theta_1102[2]) ** 2),
        "scenarios": scenarios,
    }


def build_data_summary(df_1101: pd.DataFrame, df_1102: pd.DataFrame) -> list[dict]:
    rows = []
    for date, df in [(DATES[0], df_1101), (DATES[1], df_1102)]:
        rows.append(
            {
                "date": date,
                "contracts": int(len(df)),
                "S0": float(df["S0"].iloc[0]),
                "DTE_min": float(df["DTE"].min()),
                "DTE_max": float(df["DTE"].max()),
                "tau_min": float(df["tau"].min()),
                "tau_max": float(df["tau"].max()),
                "K_min": float(df["K"].min()),
                "K_max": float(df["K"].max()),
                "log_moneyness_min": float(df["log_K_S0"].min()),
                "log_moneyness_max": float(df["log_K_S0"].max()),
                "P_mkt_min": float(df["P_mkt"].min()),
                "P_mkt_max": float(df["P_mkt"].max()),
                "iv_min": float(df["iv_market"].min()),
                "iv_max": float(df["iv_market"].max()),
                "vega_min": float(df["vega"].min()),
                "vega_max": float(df["vega"].max()),
                "r_min": float(df["r"].min()),
                "r_max": float(df["r"].max()),
                "maturity_count": int(df["DTE"].nunique()),
                "contracts_per_maturity_min": int(df.groupby("DTE").size().min()),
                "contracts_per_maturity_max": int(df.groupby("DTE").size().max()),
            }
        )
    return rows


def write_result_csvs(results: list[dict], data_summary: list[dict]) -> None:
    pd.DataFrame(data_summary).to_csv(OUT_DIR / "nvda_data_summary.csv", index=False)

    param_names = ["kappa", "lambda", "sigma", "rho", "v0"]
    param_rows = []
    for res in results:
        for date_key in ["2021_11_01", "2021_11_02"]:
            row = {
                "model": res["model"],
                "date": date_key.replace("_", "-"),
                "loss": res[f"loss_{date_key}"],
                "time_sec": res[f"time_{date_key}_sec"],
                "feller": res[f"feller_{date_key}"],
            }
            row.update({name: value for name, value in zip(param_names, res[f"theta_{date_key}"])})
            param_rows.append(row)
    pd.DataFrame(param_rows).to_csv(OUT_DIR / "nvda_parameters.csv", index=False)

    metric_rows = []
    for res in results:
        for scenario, vals in res["scenarios"].items():
            row = {"model": res["model"], "scenario": scenario}
            for key, value in vals.items():
                if key.endswith("_mre"):
                    row[key] = value
            metric_rows.append(row)
    pd.DataFrame(metric_rows).to_csv(OUT_DIR / "nvda_calibration_summary.csv", index=False)


def write_validation_details(results: list[dict], df_target: pd.DataFrame) -> None:
    rows = []
    base_cols = ["DTE", "tau", "S0", "K", "P_mkt", "iv_market", "r", "log_K_S0"]
    for res in results:
        for scenario, vals in res["scenarios"].items():
            for i, row in df_target.reset_index(drop=True).iterrows():
                item = {col: float(row[col]) for col in base_cols}
                item.update({"model": res["model"], "scenario": scenario, "contract_index": int(i + 1)})
                if "ddn_iv" in vals:
                    item["ddn_iv"] = float(vals["ddn_iv"][i])
                    item["ddn_iv_rel_err"] = abs(item["ddn_iv"] - item["iv_market"]) / (abs(item["iv_market"]) + 1e-12)
                if "ddn_price" in vals:
                    item["ddn_price"] = float(vals["ddn_price"][i])
                    item["ddn_price_rel_err"] = abs(item["ddn_price"] - item["P_mkt"]) / (abs(item["P_mkt"]) + 1e-12)
                if "ql_iv" in vals:
                    item["ql_iv"] = float(vals["ql_iv"][i])
                    item["ql_iv_rel_err"] = abs(item["ql_iv"] - item["iv_market"]) / (abs(item["iv_market"]) + 1e-12)
                if "ql_price" in vals:
                    item["ql_price"] = float(vals["ql_price"][i])
                    item["ql_price_rel_err"] = abs(item["ql_price"] - item["P_mkt"]) / (abs(item["P_mkt"]) + 1e-12)
                rows.append(item)
    pd.DataFrame(rows).to_csv(OUT_DIR / "nvda_validation_details.csv", index=False)


def write_latex_tables(results: list[dict], data_summary: list[dict]) -> None:
    d1, d2 = data_summary
    data_tex = f"""% ============================================================
% Table: NVDA Market Data Description
% ============================================================
\\begin{{table}}[htbp]
  \\centering
  \\caption{{Description of NVDA Call Option Data Used for Real-Market Calibration}}
  \\label{{tab:nvda_data}}
  \\begin{{tabular}}{{lcc}}
    \\toprule
    \\textbf{{Property}} & \\textbf{{2021-11-01 (Calibration)}} & \\textbf{{2021-11-02 (Validation)}} \\\\
    \\midrule
    \\multicolumn{{3}}{{l}}{{\\textit{{Underlying Asset}}}} \\\\
    \\quad Spot price $S_0$ (\\$)       & {d1['S0']:.2f} & {d2['S0']:.2f} \\\\
    \\quad Days to expiry (DTE)        & [{d1['DTE_min']:.0f}, {d1['DTE_max']:.0f}] & [{d2['DTE_min']:.0f}, {d2['DTE_max']:.0f}] \\\\
    \\quad Time to maturity $\\tau$ (yr)& [{d1['tau_min']:.4f}, {d1['tau_max']:.4f}] & [{d2['tau_min']:.4f}, {d2['tau_max']:.4f}] \\\\
    \\midrule
    \\multicolumn{{3}}{{l}}{{\\textit{{Option Chain (after cleaning)}}}} \\\\
    \\quad Number of valid contracts   & {d1['contracts']} & {d2['contracts']} \\\\
    \\quad Maturity buckets selected    & {d1['maturity_count']} & {d2['maturity_count']} \\\\
    \\quad Contracts per maturity       & [{d1['contracts_per_maturity_min']}, {d1['contracts_per_maturity_max']}] & [{d2['contracts_per_maturity_min']}, {d2['contracts_per_maturity_max']}] \\\\
    \\quad Strike range $K$ (\\$)       & [{d1['K_min']:.2f}, {d1['K_max']:.2f}] & [{d2['K_min']:.2f}, {d2['K_max']:.2f}] \\\\
    \\quad Log-moneyness $\\ln(K/S_0)$  & [{d1['log_moneyness_min']:+.4f}, {d1['log_moneyness_max']:+.4f}] & [{d2['log_moneyness_min']:+.4f}, {d2['log_moneyness_max']:+.4f}] \\\\
    \\quad Market IV range             & [{d1['iv_min']:.4f}, {d1['iv_max']:.4f}] & [{d2['iv_min']:.4f}, {d2['iv_max']:.4f}] \\\\
    \\quad Vega range                  & [{d1['vega_min']:.4f}, {d1['vega_max']:.4f}] & [{d2['vega_min']:.4f}, {d2['vega_max']:.4f}] \\\\
    \\midrule
    \\multicolumn{{3}}{{l}}{{\\textit{{Risk-Free Rate (continuous, interpolated)}}}} \\\\
    \\quad Matched $r$ range           & [{d1['r_min']:.6f}, {d1['r_max']:.6f}] & [{d2['r_min']:.6f}, {d2['r_max']:.6f}] \\\\
    \\quad Rate source                 & \\multicolumn{{2}}{{c}}{{US Par Yield Curve (Federal Reserve, 2020--2023)}} \\\\
    \\quad Interpolation method        & \\multicolumn{{2}}{{c}}{{Cubic spline on 1mo/2mo/3mo/6mo/1yr nodes}} \\\\
    \\midrule
    \\multicolumn{{3}}{{l}}{{\\textit{{Cleaning Criteria}}}} \\\\
    \\quad Option type                 & \\multicolumn{{2}}{{c}}{{Call only}} \\\\
    \\quad DTE filter                  & \\multicolumn{{2}}{{c}}{{40--300 days}} \\\\
    \\quad Log-moneyness filter        & \\multicolumn{{2}}{{c}}{{$\\ln(K/S_0) \\in [-0.15,\\;0.20]$}} \\\\
    \\quad Volume filter               & \\multicolumn{{2}}{{c}}{{Positive call volume required}} \\\\
    \\quad Final selection             & \\multicolumn{{2}}{{c}}{{100 contracts, balanced by maturity and centered around $S_0$ in strike}} \\\\
    \\bottomrule
  \\end{{tabular}}
\\end{{table}}
"""
    (OUT_DIR / "table_nvda_data.tex").write_text(data_tex)

    heston = next(res for res in results if res["model"] == "Heston_Project")
    zhang = next(res for res in results if res["model"] == "Zhang_realize")

    def metric(res: dict, scenario: str, key: str) -> float:
        return res["scenarios"][scenario].get(key, float("nan"))

    calibration_tex = f"""% ============================================================
% Table: NVDA Same-Day and Cross-Day Calibration
% ============================================================
\\begin{{table}}[htbp]
  \\centering
  \\caption{{Same-Day and Cross-Day Calibration Performance on NVDA Options (2021-11-01 $\\rightarrow$ 2021-11-02, $M={N_CONTRACTS}$ contracts)}}
  \\label{{tab:nvda_calibration}}
  \\setlength{{\\tabcolsep}}{{6pt}}
  \\begin{{tabular}}{{llcc}}
    \\toprule
    \\textbf{{Scenario}} & \\textbf{{Metric}} & \\textbf{{Heston Project}} & \\textbf{{Zhang (2025)}} \\\\
    \\midrule
    \\multicolumn{{4}}{{l}}{{\\textit{{Same-Day: parameters calibrated on 2021-11-02 and evaluated on 2021-11-02}}}} \\\\
    \\quad Same-day & Native model IV MRE & {pct_tex(metric(heston, 'same_day', 'ddn_iv_mre'))} & {pct_tex(metric(zhang, 'same_day', 'ddn_iv_mre'))} \\\\
    \\quad Same-day & Native model price MRE & n/a & {pct_tex(metric(zhang, 'same_day', 'ddn_price_mre'))} \\\\
    \\quad Same-day & QuantLib IV MRE & {pct_tex(metric(heston, 'same_day', 'ql_iv_mre'))} & {pct_tex(metric(zhang, 'same_day', 'ql_iv_mre'))} \\\\
    \\quad Same-day & QuantLib price MRE & {pct_tex(metric(heston, 'same_day', 'ql_price_mre'))} & {pct_tex(metric(zhang, 'same_day', 'ql_price_mre'))} \\\\
    \\midrule
    \\multicolumn{{4}}{{l}}{{\\textit{{Cross-Day: parameters calibrated on 2021-11-01 and evaluated on 2021-11-02}}}} \\\\
    \\quad Cross-day & Native model IV MRE & {pct_tex(metric(heston, 'cross_day', 'ddn_iv_mre'))} & {pct_tex(metric(zhang, 'cross_day', 'ddn_iv_mre'))} \\\\
    \\quad Cross-day & Native model price MRE & n/a & {pct_tex(metric(zhang, 'cross_day', 'ddn_price_mre'))} \\\\
    \\quad Cross-day & QuantLib IV MRE & {pct_tex(metric(heston, 'cross_day', 'ql_iv_mre'))} & {pct_tex(metric(zhang, 'cross_day', 'ql_iv_mre'))} \\\\
    \\quad Cross-day & QuantLib price MRE & {pct_tex(metric(heston, 'cross_day', 'ql_price_mre'))} & {pct_tex(metric(zhang, 'cross_day', 'ql_price_mre'))} \\\\
    \\midrule
    \\multicolumn{{4}}{{l}}{{\\textit{{Degradation from same-day to cross-day}}}} \\\\
    \\quad $\\Delta$ native IV MRE & Cross-day minus same-day & {100.0 * (metric(heston, 'cross_day', 'ddn_iv_mre') - metric(heston, 'same_day', 'ddn_iv_mre')):+.2f} pp & {100.0 * (metric(zhang, 'cross_day', 'ddn_iv_mre') - metric(zhang, 'same_day', 'ddn_iv_mre')):+.2f} pp \\\\
    \\bottomrule
  \\end{{tabular}}
  \\begin{{tablenotes}}
    \\small
    \\item \\textit{{Note:}} IV MRE and price MRE are mean relative errors. The Heston Project model is calibrated directly in implied-volatility space with Vega weighting and a Feller penalty. The Zhang baseline is calibrated in normalised price space; its native IV MRE is computed by converting DDN-predicted prices to Black--Scholes implied volatilities. QuantLib rows re-price the selected NVDA contracts with the calibrated Heston parameters.
  \\end{{tablenotes}}
\\end{{table}}
"""
    (OUT_DIR / "table_nvda_calibration.tex").write_text(calibration_tex)

    param_lines = [
        "% ============================================================",
        "% Table: NVDA Calibrated Parameters",
        "% ============================================================",
        "\\begin{table}[htbp]",
        "  \\centering",
        "  \\caption{Calibrated Heston Parameters on NVDA Options}",
        "  \\label{tab:nvda_parameters}",
        "  \\begin{tabular}{llrrrrr}",
        "    \\toprule",
        "    \\textbf{Model} & \\textbf{Calibration Date} & $\\kappa$ & $\\theta$ & $\\sigma$ & $\\rho$ & $v_0$ \\\\",
        "    \\midrule",
    ]
    for res in results:
        model_label = res["model"].replace("_", "\\_")
        for date_key, date_label in [("2021_11_01", "2021-11-01"), ("2021_11_02", "2021-11-02")]:
            theta = res[f"theta_{date_key}"]
            param_lines.append(f"    {model_label} & {date_label} & " + " & ".join(f"{v:.6f}" for v in theta) + " \\\\")
    param_lines.extend(["    \\bottomrule", "  \\end{tabular}", "\\end{table}", ""])
    (OUT_DIR / "table_nvda_parameters.tex").write_text("\n".join(param_lines))


yield_data = pd.read_csv(ROOT / "data" / "par-yield-curve-rates-2020-2023.csv")
df_nvda_1101 = load_and_clean_nvda(yield_data, "2021-11-01")
df_nvda_1102 = load_and_clean_nvda(yield_data, "2021-11-02")

df_nvda_1101.to_csv(OUT_DIR / "nvda_selected_2021-11-01.csv", index=False)
df_nvda_1102.to_csv(OUT_DIR / "nvda_selected_2021-11-02.csv", index=False)

print("Running Heston_Project NVDA calibration...")
heston_result = run_heston_project(df_nvda_1101, df_nvda_1102)
print("Running Zhang_realize NVDA calibration...")
zhang_result = run_zhang_realize(df_nvda_1101, df_nvda_1102)

results = [heston_result, zhang_result]
data_summary = build_data_summary(df_nvda_1101, df_nvda_1102)
write_result_csvs(results, data_summary)
write_validation_details(results, df_nvda_1102)
write_latex_tables(results, data_summary)

(OUT_DIR / "nvda_calibration_results.json").write_text(json.dumps({"data_summary": data_summary, "results": results}, indent=2))

summary_df = pd.read_csv(OUT_DIR / "nvda_calibration_summary.csv")
print("\nNVDA calibration summary:")
print(summary_df.to_string(index=False))
print("\nWrote LaTeX tables:")
print(OUT_DIR / "table_nvda_data.tex")
print(OUT_DIR / "table_nvda_calibration.tex")
print(OUT_DIR / "table_nvda_parameters.tex")
